# 🌸 TohoStudio: Google Colab L4 GPU メディア生成サーバー
### （SDXL 1.0 画像 / AudioLDM 2 効果音 / MusicGen Medium 音楽 / AnimateDiff 動画）

このノートブックは、Google Colab の **NVIDIA L4 GPU（VRAM 24GB）** を活用し、**汎用的な高品質メディア生成（高精細イラスト・写真、本物の効果音SFX、豊かなBGM、動画）** を手元の TohoStudio アプリに提供するサーバーです。

### 🚀 主な特徴 (L4 GPU / 24GB VRAM 特化)
- **画像生成**: `stabilityai/stable-diffusion-xl-base-1.0` (SDXL 1.0)
  - 1024px ネイティブ解像度、30ステップ推論、Guidance Scale 7.5
  - アスペクト比（16:9 / 1:1 / 9:16）および除外プロンプト（Negative Prompt）完全対応
  - アニメ調、実写写真、3D/CG、幻想的風景などあらゆるジャンルを極上の品質で生成
- **効果音生成 (SE)**: `cvssp/audioldm2` (AudioLDM 2)
  - 音響・SFX特化モデルによる本物の単発効果音・環境音生成（レーザー、爆発、足音、魔法音等）
- **BGM音楽生成**: `facebook/musicgen-medium` (MusicGen Medium)
  - 高音質な音楽生成（ジャズ、和風、エレクトロ、オーケストラ、Lo-Fi等）
- **動画生成**: `AnimateDiff` + `Stable Diffusion` (MP4 形式)
- **メモリ管理**: 24GB VRAM を最大限活かしつつ、各生成後に GPU キャッシュを自動解放して安定動作を保証


In [ ]:
# [1/4] GPU の確認（NVIDIA L4 GPU 24GB 推奨）
!nvidia-smi


In [ ]:
# [2/4] 依存ライブラリと Cloudflare Tunnel のインストール
!pip install -q fastapi uvicorn pydantic diffusers transformers accelerate torch scipy soundfile imageio[ffmpeg] safetensors invisible-watermark
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ L4 GPU 高品位生成ライブラリとトンネルツールの準備が完了しました！")


In [ ]:
# [3/4] FastAPI サーバープログラムの作成 (SDXL 1.0 + AudioLDM 2 + MusicGen Medium)
server_code = '''
import io, os, tempfile, torch, scipy.io.wavfile, gc
from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel
from typing import Optional
from diffusers import AutoPipelineForText2Image, AudioLDM2Pipeline, AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video
from transformers import AutoProcessor, MusicgenForConditionalGeneration

app = FastAPI(title="TohoStudio L4 GPU Media Bridge", version="3.0.0")
device = "cuda" if torch.cuda.is_available() else "cpu"

# オンデマンドパイプライン保持変数
pipe_img = None
pipe_se = None
music_processor = None
music_model = None
pipe_video = None

def clean_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_image_pipe():
    global pipe_img
    if pipe_img is None:
        print("🎨 Loading SDXL 1.0 Image Pipeline (L4 24GB VRAM Optimized)..." )
        pipe_img = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        ).to(device)
    return pipe_img

def get_se_pipe():
    global pipe_se
    if pipe_se is None:
        print("🔊 Loading AudioLDM 2 Sound Effect Pipeline..." )
        pipe_se = AudioLDM2Pipeline.from_pretrained(
            "cvssp/audioldm2",
            torch_dtype=torch.float16
        ).to(device)
    return pipe_se

def get_music_pipe():
    global music_processor, music_model
    if music_model is None:
        print("🎵 Loading MusicGen Medium BGM Pipeline..." )
        music_processor = AutoProcessor.from_pretrained("facebook/musicgen-medium")
        music_model = MusicgenForConditionalGeneration.from_pretrained(
            "facebook/musicgen-medium",
            torch_dtype=torch.float16
        ).to(device)
    return music_processor, music_model

def get_video_pipe():
    global pipe_video
    if pipe_video is None:
        print("🎬 Loading AnimateDiff Video Pipeline..." )
        adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
        pipe_video = AnimateDiffPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            motion_adapter=adapter,
            torch_dtype=torch.float16
        ).to(device)
        pipe_video.scheduler = DDIMScheduler.from_config(pipe_video.scheduler.config, beta_schedule="linear", steps_offset=1)
    return pipe_video

@app.get("/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    vram_free = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    return {
        "status": "READY",
        "gpu": gpu_name,
        "vram_free_gb": round(vram_free, 2),
        "capabilities": ["image (SDXL 1.0)", "se (AudioLDM 2)", "bgm (MusicGen Medium)", "video (AnimateDiff)"]
    }

class MediaPromptRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = None
    width: Optional[int] = 1344
    height: Optional[int] = 768
    seed: Optional[int] = -1
    duration_seconds: Optional[float] = 8.0

# [1] 高精細汎用画像生成 (SDXL 1.0 / 30 steps / Guidance 7.5)
@app.post("/v1/generate/image")
def generate_image(req: MediaPromptRequest):
    pipe = get_image_pipe()
    w = max(512, min(1536, (req.width // 8) * 8)) if req.width else 1344
    h = max(512, min(1536, (req.height // 8) * 8)) if req.height else 768
    
    # 汎用ネガティブプロンプトの構成
    base_neg = "ugly, deformed, disfigured, poor quality, bad anatomy, bad hands, missing fingers, cropped, watermark, text"
    if req.negative_prompt and len(req.negative_prompt.strip()) > 0:
        final_neg = f"{req.negative_prompt.strip()}, {base_neg}"
    else:
        final_neg = base_neg

    gen = None
    if req.seed is not None and req.seed >= 0:
        gen = torch.Generator(device=device).manual_seed(req.seed)

    print(f"[Image] Generating SDXL ({w}x{h}). Prompt: {req.prompt[:60]}...")
    image = pipe(
        prompt=req.prompt,
        negative_prompt=final_neg,
        width=w,
        height=h,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=gen
    ).images[0]

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    clean_gpu()
    return Response(content=buf.getvalue(), media_type="image/png")

# [2] BGM 音楽生成 (MusicGen Medium / 高音質)
@app.post("/v1/generate/music")
@app.post("/v1/generate/bgm")
def generate_bgm(req: MediaPromptRequest):
    proc, model = get_music_pipe()
    inputs = proc(text=[req.prompt], padding=True, return_tensors="pt").to(device)
    dur = min(30.0, max(2.0, req.duration_seconds if req.duration_seconds else 10.0))
    max_tokens = int(dur * 50)
    
    print(f"[BGM] Generating MusicGen ({dur}s). Prompt: {req.prompt[:60]}...")
    with torch.no_grad():
        audio_values = model.generate(**inputs, max_new_tokens=max_tokens)
    
    sampling_rate = model.config.audio_encoder.sampling_rate
    wav_data = audio_values[0, 0].cpu().numpy()
    buf = io.BytesIO()
    scipy.io.wavfile.write(buf, rate=sampling_rate, data=wav_data)
    clean_gpu()
    return Response(content=buf.getvalue(), media_type="audio/wav")

# [3] 効果音 (SE) 生成 (AudioLDM 2 特化音響モデル)
@app.post("/v1/generate/se")
def generate_se(req: MediaPromptRequest):
    pipe = get_se_pipe()
    dur = min(10.0, max(1.0, req.duration_seconds if req.duration_seconds else 2.5))
    
    print(f"[SE] Generating AudioLDM2 ({dur}s). Prompt: {req.prompt[:60]}...")
    with torch.no_grad():
        audio = pipe(
            prompt=req.prompt,
            num_inference_steps=25,
            audio_length_in_s=dur
        ).audios[0]

    buf = io.BytesIO()
    scipy.io.wavfile.write(buf, rate=16000, data=audio)
    clean_gpu()
    return Response(content=buf.getvalue(), media_type="audio/wav")

# [4] 動画 (MP4) 生成 (AnimateDiff)
@app.post("/v1/generate/video")
def generate_video(req: MediaPromptRequest):
    pipe = get_video_pipe()
    w = max(384, min(768, (req.width // 8) * 8)) if req.width else 512
    h = max(384, min(768, (req.height // 8) * 8)) if req.height else 512
    neg = req.negative_prompt if (req.negative_prompt and len(req.negative_prompt.strip()) > 0) else "worst quality, low quality, deformed, blurry"

    print(f"[Video] Generating AnimateDiff ({w}x{h}). Prompt: {req.prompt[:60]}...")
    output = pipe(
        prompt=req.prompt,
        negative_prompt=neg,
        width=w,
        height=h,
        num_frames=16,
        guidance_scale=7.5,
        num_inference_steps=25,
    )
    frames = output.frames[0]
    temp_file = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    temp_path = temp_file.name
    temp_file.close()

    export_to_video(frames, temp_path, fps=8)
    with open(temp_path, "rb") as f:
        video_data = f.read()
    os.remove(temp_path)
    clean_gpu()
    return Response(content=video_data, media_type="video/mp4")
'''

with open("app.py", "w") as f:
    f.write(server_code)
print("✅ L4 GPU 高品位サーバーコード（app.py）を書き出しました。")


In [ ]:
# [4/4] サーバー起動と外部トンネルURLの発行
import subprocess, time, re

# 既存のプロセスを終了
!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

# uvicorn サーバーを起動
print("L4 GPU サーバーを起動しています...")
server_proc = subprocess.Popen(["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(30):
    line = server_proc.stdout.readline()
    if line:
        print(line.strip())
    if "Application startup complete" in line:
        break
    time.sleep(0.5)

# Cloudflare Tunnel を起動して一時 URL を取得
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if public_url:
    print("\n===================================================================")
    print("🎉 Colab L4 GPU 4大メディア高品位生成サーバーが正常に起動しました！")
    print(f"👉 接続先 URL: {public_url}")
    print("===================================================================\n")
else:
    print("トンネルURLの取得に失敗しました。")
